<a href="https://colab.research.google.com/github/uyx7uy/ds2002-fa26/blob/main/2026_09_18_Pandas_Challenge_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total revenue was ${total_revenue:.2f}.")
print(f"Total units sold was {total_units}.")

Total revenue was $8520.00.
Total units sold was 783.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
by_category = df.groupby('category', as_index=False)['revenue'].sum()
by_category['share_pct'] = by_category['revenue'] / total_revenue * 100
by_category = by_category.sort_values('revenue', ascending=False)

by_category

,category,revenue,share_pct
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
by_vendor = df.groupby('vendor_id').agg(
    avg_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
)

by_vendor = by_vendor.sort_values('avg_revenue', ascending=False)
by_vendor

,avg_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()
merch_share = merch_revenue / total_revenue * 100

print(f"Merch made up {merch_share:.1f}% of total revenue.")# TODO

Merch made up 20.8% of total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

print(len(df), len(joined))
print(df['revenue'].sum(), joined['revenue'].sum())

missing_vendor = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
print(missing_vendor)

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')

400 400
8520.0 8520.0
['V-18']


The unmatched vendor was V-18. I kept those rows in the data because the sales still happened, but I labeled the vendor name as Unknown vendor so the revenue would not be dropped.



**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
pivot = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell the vendors to prepare the most for Food because it brought in $4293.00, which was about 50.4% of all revenue. Merch was also important at $1771.50, or 20.8% of revenue, so vendors should make sure they have enough food and merch ready for the next game. The Unknown vendor also had the highest total vendor revenue at $2349.00, so I would want to identify that vendor before making a final recommendation.

b) The least trustworthy answer is the vendor-name report because one vendor ID, V-18, did not match the lookup table. I kept it as Unknown vendor so the revenue was not removed, but it makes that part of the report less clear because I do not know which real vendor earned that revenue.